# 📈 Notebook 04 — Model Training (Regression)

**Purpose**: Train multiple regression models to predict `cycle_degradation`, tune hyperparameters, select best model.

**Theory**:
- Lecture 04: Linear Regression (foundational baseline)
- Lecture 06: Random Forest (bagging reduces variance)
- Lecture 07: XGBoost (best for structured tabular data with L1/L2 regularization)

**Target**: `cycle_degradation` (log1p transformed)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, sys, time, warnings
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
os.makedirs('../plots', exist_ok=True)
os.makedirs('../models', exist_ok=True)
sys.path.insert(0, '..')
from src.feature_engineer import engineer_all_features
print('Ready.')

In [ ]:
# Load and prepare data
df = pd.read_csv('../data/nev_battery_charging.csv').drop(columns=['timestamp']).drop_duplicates()
n = len(df); t_end, v_end = int(n*0.70), int(n*0.85)
targets = ['cycle_degradation','over_temp_flag','over_voltage_flag']

df_train, df_val, df_test = df.iloc[:t_end], df.iloc[t_end:v_end], df.iloc[v_end:]
baseline_ir = df_train['internal_resistance'].iloc[0]

# Engineer features
for d in [df_train, df_val, df_test]:
    d_copy = d  # reference

df_train = engineer_all_features(df_train, baseline_ir, ['battery_temp'])
df_val = engineer_all_features(df_val, baseline_ir, ['battery_temp'])
df_test = engineer_all_features(df_test, baseline_ir, ['battery_temp'])

# Load selected features from NB03
try:
    with open('../models/feature_columns.json') as f:
        selected_features = json.load(f)
    print(f'Loaded {len(selected_features)} selected features from NB03')
except FileNotFoundError:
    selected_features = [c for c in df_train.columns if c not in targets]
    print(f'No feature_columns.json found, using all {len(selected_features)} features')

# Separate X and y
X_train = df_train[[c for c in selected_features if c in df_train.columns]]
X_val = df_val[[c for c in selected_features if c in df_val.columns]]
X_test = df_test[[c for c in selected_features if c in df_test.columns]]
y_train = df_train['cycle_degradation']; y_val = df_val['cycle_degradation']; y_test = df_test['cycle_degradation']

# Scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Log transform target
y_train_log = np.log1p(y_train); y_val_log = np.log1p(y_val); y_test_log = np.log1p(y_test)
print(f'X_train: {X_train_s.shape}, X_val: {X_val_s.shape}, X_test: {X_test_s.shape}')

## 1. Baseline — Linear Regression

**Theory**: Lecture 04 — Linear regression minimizes MSE cost function. R² tells how much variance is explained. If R² < 0.5, we need non-linear models.

In [ ]:
# Linear Regression
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_s, y_train_log)
lr_time = time.time() - t0

y_pred_lr = np.expm1(lr.predict(X_val_s))
mae_lr = mean_absolute_error(y_val, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
r2_lr = r2_score(y_val, y_pred_lr)

print(f'Linear Regression Results (Validation Set):')
print(f'  MAE:  {mae_lr:.8f}')
print(f'  RMSE: {rmse_lr:.8f}')
print(f'  R²:   {r2_lr:.4f}')
print(f'  Time: {lr_time:.3f}s')

if r2_lr < 0.5:
    print('\n→ R² < 0.5: Linear relationships insufficient. Non-linear models needed.')
else:
    print('\n→ R² > 0.5: Some linear relationship exists. Still try non-linear for improvement.')

## 2. Random Forest Regressor

**Theory**: Lecture 06 — Random Forest uses bagging with random feature selection. More trees = better with diminishing returns.

**Critical**: Using `TimeSeriesSplit` for CV, NOT default KFold (which shuffles = data leakage).

In [ ]:
# Random Forest with GridSearch + TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

rf_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 0.5]
}

t0 = time.time()
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_grid, cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs_rf.fit(X_train_s, y_train_log)
rf_time = time.time() - t0

best_rf = gs_rf.best_estimator_
print(f'Best RF params: {gs_rf.best_params_}')
print(f'Best CV RMSE: {-gs_rf.best_score_:.6f}')

y_pred_rf = np.expm1(best_rf.predict(X_val_s))
mae_rf = mean_absolute_error(y_val, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
r2_rf = r2_score(y_val, y_pred_rf)
print(f'\nRandom Forest Results (Validation Set):')
print(f'  MAE:  {mae_rf:.8f}')
print(f'  RMSE: {rmse_rf:.8f}')
print(f'  R²:   {r2_rf:.4f}')
print(f'  Time: {rf_time:.1f}s')

In [ ]:
# Feature Importance from Random Forest
importances = best_rf.feature_importances_
feat_names = list(X_train.columns)
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(idx)))
ax.barh(range(len(idx)), importances[idx][::-1], color=colors)
ax.set_yticks(range(len(idx)))
ax.set_yticklabels([feat_names[i] for i in idx][::-1])
ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
ax.set_title('Random Forest — Feature Importances for Regression', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features:')
for i in range(min(5, len(idx))):
    print(f'  {feat_names[idx[i]]}: {importances[idx[i]]:.4f}')

## 3. XGBoost Regressor

**Theory**: Lecture 07 — XGBoost uses sequential boosting where each model corrects residuals. L1 (reg_alpha) and L2 (reg_lambda) regularization prevent overfitting. Early stopping halts when validation loss stops improving.

**Why XGBoost**: Lecture 07 comparison — best for structured/tabular data on moderate datasets. 1900 rows of battery sensor data is exactly this use case.

In [ ]:
# XGBoost with GridSearch + TimeSeriesSplit
xgb_grid = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1, 5, 10]
}

t0 = time.time()
gs_xgb = GridSearchCV(
    XGBRegressor(objective='reg:squarederror', random_state=42, eval_metric='rmse', verbosity=0),
    xgb_grid, cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs_xgb.fit(X_train_s, y_train_log)
print(f'Best XGB params: {gs_xgb.best_params_}')
print(f'Best CV RMSE: {-gs_xgb.best_score_:.6f}')

In [ ]:
# Retrain best XGBoost with early stopping for training curve
best_xgb = XGBRegressor(
    **gs_xgb.best_params_,
    objective='reg:squarederror',
    random_state=42,
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbosity=0
)
best_xgb.fit(
    X_train_s, y_train_log,
    eval_set=[(X_train_s, y_train_log), (X_val_s, y_val_log)],
    verbose=False
)
xgb_time = time.time() - t0

y_pred_xgb = np.expm1(best_xgb.predict(X_val_s))
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
r2_xgb = r2_score(y_val, y_pred_xgb)
print(f'\nXGBoost Results (Validation Set):')
print(f'  MAE:  {mae_xgb:.8f}')
print(f'  RMSE: {rmse_xgb:.8f}')
print(f'  R²:   {r2_xgb:.4f}')
print(f'  Time: {xgb_time:.1f}s')

In [ ]:
# XGBoost training curve
evals = best_xgb.evals_result()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(evals['validation_0']['rmse'], label='Training RMSE', color='#2196F3', linewidth=1.5)
ax.plot(evals['validation_1']['rmse'], label='Validation RMSE', color='#F44336', linewidth=1.5)
ax.set_xlabel('Boosting Round', fontsize=12)
ax.set_ylabel('RMSE (log-scale)', fontsize=12)
ax.set_title('XGBoost — Training vs Validation RMSE Over Boosting Rounds', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/xgb_training_curve_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best iteration: {best_xgb.best_iteration}')
print(f'Final train RMSE: {evals["validation_0"]["rmse"][-1]:.6f}')
print(f'Final val RMSE:   {evals["validation_1"]["rmse"][-1]:.6f}')

## 4. Model Comparison

In [ ]:
# Comparison table
results = {
    'LinearRegression': {'MAE': mae_lr, 'RMSE': rmse_lr, 'R2': r2_lr, 'Time(s)': lr_time},
    'RandomForest': {'MAE': mae_rf, 'RMSE': rmse_rf, 'R2': r2_rf, 'Time(s)': rf_time},
    'XGBoost': {'MAE': mae_xgb, 'RMSE': rmse_xgb, 'R2': r2_xgb, 'Time(s)': xgb_time}
}
results_df = pd.DataFrame(results).T
print('\n=== Regression Model Comparison (Validation Set) ===')
print(results_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = list(results.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, metric, higher_better in zip(axes, ['MAE','RMSE','R2'], [False,False,True]):
    vals = [results[m][metric] for m in models]
    bars = ax.bar(models, vals, color=colors, edgecolor='white', linewidth=2)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    best_idx = vals.index(max(vals)) if higher_better else vals.index(min(vals))
    bars[best_idx].set_edgecolor('#F44336'); bars[best_idx].set_linewidth(3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{v:.6f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Regression Model Comparison', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../plots/regression_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Learning Curve Analysis

**Theory**: Lecture 01 — Overfitting = good on train, bad on new data. Underfitting = too simple. Learning curve visualises which condition applies.

In [ ]:
# Select best model
best_name = min(results, key=lambda k: results[k]['RMSE'])
best_model = {'LinearRegression': lr, 'RandomForest': best_rf, 'XGBoost': best_xgb}[best_name]
print(f'Best model: {best_name} (RMSE={results[best_name]["RMSE"]:.8f})')

# Learning curve
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_s, y_train_log,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(12, 6))
train_mean = -train_scores.mean(axis=1); val_mean = -val_scores.mean(axis=1)
train_std = train_scores.std(axis=1); val_std = val_scores.std(axis=1)

ax.plot(train_sizes, train_mean, 'o-', color='#2196F3', label='Training RMSE', linewidth=2)
ax.plot(train_sizes, val_mean, 'o-', color='#F44336', label='Validation RMSE', linewidth=2)
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='#2196F3')
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='#F44336')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('RMSE (log-scale)', fontsize=12)
ax.set_title(f'Learning Curve — {best_name}', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/learning_curve_regression.png', dpi=150, bbox_inches='tight')
plt.show()

if val_mean[-1] - train_mean[-1] > 0.5 * train_mean[-1]:
    print('→ Large gap: possible OVERFITTING. Consider more regularization.')
elif train_mean[-1] > 0.01:
    print('→ Both curves high: possible UNDERFITTING. Try more complex features.')
else:
    print('→ Curves converging: GOOD generalization.')

## 6. Save Best Model

In [ ]:
# Save best regression model
joblib.dump(best_model, '../models/regression_model.pkl')
print(f'Saved: models/regression_model.pkl ({best_name})')

# Save scaler
joblib.dump({'scaler': scaler, 'imputer': None, 'fit_ir_baseline': baseline_ir,
             'columns_to_drop': ['battery_temp']}, '../models/preprocessor_pipeline.pkl')
print('Saved: models/preprocessor_pipeline.pkl')

# Save results
json_results = {k: {mk: float(mv) for mk, mv in v.items()} for k, v in results.items()}
with open('../models/regression_results.json', 'w') as f:
    json.dump(json_results, f, indent=2)
print('Saved: models/regression_results.json')

## 📋 Regression Training Summary

| Model | MAE | RMSE | R² | Training Time |
|-------|-----|------|----|---------------|
| Linear Regression | Baseline | Baseline | Baseline | Fast |
| Random Forest | GridSearchCV | TimeSeriesSplit | Bagging | Medium |
| XGBoost | GridSearchCV | Early Stopping | Boosting | Medium-High |

**Selected model saved to `models/regression_model.pkl`**